In [2]:
# from comet_ml import Experiment
# from comet_ml.integration.pytorch import log_model

import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import torch.nn.functional as F
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import pickle as pkl

import matplotlib.pyplot as plt
import numpy as np
import io, os
from tqdm import tqdm

from PIL import Image
from torchvision import models 
from torchvision.models import resnet18

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import KBinsDiscretizer
import statsmodels.api as sm


### Dataset definition

In [23]:
class EnsembleDataset(Dataset):
    def __init__(self, curve_dict, target_df,img_directory = 'data/curve_imgs/', split='train', sequence_len=40, 
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        self.target_df = pd.concat([self.target_df, self.one_hot], axis=1)

        self.img_directory = img_directory
        self.split = split
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((224, 224)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])

        #Implementation of train test split
        if self.split == 'train':
            self.target_df = self.target_df[self.target_df['split']=='train']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'val':
            self.target_df = self.target_df[self.target_df['split']=='val']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'test':
            self.target_df = self.target_df[self.target_df['split']=='test']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        else:
            raise NotImplementedError
        
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #gene info processing
        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]
        gene_type = torch.tensor(row[self.one_hot.columns].values, dtype=torch.float32)

        #target data retrieval
        # Extract values from the dataframe
        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.long)
        igi_call = torch.tensor(row['Igi_call_quant'].values[0], dtype=torch.long)
        igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.long)
        igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.long)

        return curve_img, sequence_normalized, gene_type, igi_call, igi_fp, igi_fn, target, curve_idx

class ViTFusionDataset(Dataset):
    def __init__(self, curve_dict, target_df,img_directory = 'data/curve_imgs/', split='train', sequence_len=40, 
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        target_ls = ['target_' + t for t in ['S gene','N gene','E gene','RnaseP','MS2','ORF1ab']]

        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        
        if self.one_hot.shape[1] != 6:
            for target in target_ls:
                if target not in self.one_hot.columns:
                    self.one_hot[target] = False
        
        self.target_df = pd.concat([self.target_df, self.one_hot], axis=1)

        self.img_directory = img_directory
        self.split = split
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((224, 224)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])

        #Implementation of train test split
        if self.split == 'train':
            self.target_df = self.target_df[self.target_df['split']=='train']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'val':
            self.target_df = self.target_df[self.target_df['split']=='val']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'test':
            self.target_df = self.target_df[self.target_df['split']=='test']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        else:
            raise NotImplementedError
        
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]
        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #gene info processing
        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]
        gene_type = torch.tensor(row[self.one_hot.columns].values, dtype=torch.float32)

        #target data retrieval
        # Extract values from the dataframe
        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.long)
        igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.long)
        igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.long)

        # Create a 3-dimensional vector
        vector = [target, igi_fp, igi_fn]
        #target = self.target_df.loc[self.target_df['curve_idx'] == curve_idx, 'groundtruth_target'].values[0]

        return curve_img, sequence_normalized, gene_type, vector, curve_idx

class ImageSequenceGeneDataset(Dataset):
    def __init__(self, curve_dict, target_df,img_directory = 'data/curve_imgs/', split='train', sequence_len=40, 
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        self.target_df = pd.concat([self.target_df, self.one_hot], axis=1)

        self.img_directory = img_directory
        self.split = split
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((128, 128)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])

        #Implementation of train test split
        if self.split == 'train':
            self.target_df = self.target_df[self.target_df['split']=='train']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'val':
            self.target_df = self.target_df[self.target_df['split']=='val']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'test':
            self.target_df = self.target_df[self.target_df['split']=='test']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        else:
            raise NotImplementedError
        
        # if self.train:
        #     self.target_df = self.target_df[self.target_df['split']=='train']
        #     self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        # else:
        #     self.target_df = self.target_df[self.target_df['split']=='val']
        #     self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #gene info processing
        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]
        gene_type = torch.tensor(row[self.one_hot.columns].values, dtype=torch.float32)

        #target data retrieval
        # Extract values from the dataframe
        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.long)
        igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.long)
        igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.long)

        # Create a 3-dimensional vector
        vector = [target, igi_fp, igi_fn]
        #target = self.target_df.loc[self.target_df['curve_idx'] == curve_idx, 'groundtruth_target'].values[0]

        return curve_img, sequence_normalized, gene_type, vector, curve_idx

class ImageSequenceDataset(Dataset):
    def __init__(self, curve_dict, target_df,img_directory = 'data/curve_imgs/', split = 'train', sequence_len=40, 
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df
        self.img_directory = img_directory
        self.split = split
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((128, 128)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])

        #Implementation of train test split
        if self.split == 'train':
            self.target_df = self.target_df[self.target_df['split']=='train']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'val':
            self.target_df = self.target_df[self.target_df['split']=='val']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'test':
            self.target_df = self.target_df[self.target_df['split']=='test']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        else:
            raise NotImplementedError
        
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #target data retrieval
        target = self.target_df.loc[self.target_df['curve_idx'] == curve_idx, 'groundtruth_target'].values[0]

        return curve_img, sequence_normalized, torch.tensor(target, dtype=torch.long), curve_idx


### Model definition

In [4]:

class FusionModel(nn.Module):
    def __init__(self, input_size, hidden_size, latent_dim, sequence_length, num_layers=5):
        super(FusionModel, self).__init__()

        self.latent_dim = latent_dim
        
        # Image processing via EfficientNet_V2_L
        self.effnet = models.efficientnet_v2_l(pretrained=True)
        num_ftrs = self.effnet.classifier[1].in_features
        self.effnet.classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, 512)

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(1024, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, image, sequence):
        # Image processing
        img_latent = self.effnet(image)

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        fusion = torch.cat((img_latent, seq_latent), dim=1)
        output = self.fc(fusion)
        return output, img_latent, seq_latent

class ImageModel(nn.Module):
    def __init__(self, input_size, hidden_size, latent_dim):
        super(ImageModel, self).__init__()

        self.latent_dim = latent_dim
        
        # Image processing via EfficientNet_V2_L
        self.effnet = models.efficientnet_v2_l(pretrained=True)
        num_ftrs = self.effnet.classifier[1].in_features
        self.effnet.classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # FC Layers
        self.fc = nn.Sequential(
            nn.Linear(self.latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, image, sequence):
        # Image processing
        img_latent = self.effnet(image)
        output = self.fc(img_latent)
        return output


class FusionwGeneModel(nn.Module):
    def __init__(self, input_size, hidden_size, latent_dim, sequence_length, num_layers=5, genes = 6, num_heads=3):
        super(FusionwGeneModel, self).__init__()

        self.latent_dim = latent_dim
        
        # Image processing via EfficientNet_V2_L
        # TODO change to true
        self.effnet = models.efficientnet_v2_l(pretrained=True)
        num_ftrs = self.effnet.classifier[1].in_features
        self.effnet.classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes
        neural_net_input = self.latent_dim*2 + genes

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
            # nn.Linear(64, 1),
            # nn.Sigmoid()
        )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])


    def forward(self, image, sequence, genes):
        # Image processing
        img_latent = self.effnet(image)

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1)), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = [torch.sigmoid(head(output)) for head in self.heads]

        return outputs

class ViTFusionModel(nn.Module):
    def __init__(self, input_size, hidden_size, latent_dim, sequence_length, num_layers=5, genes = 6, num_heads=3, delta=16):
        super(ViTFusionModel, self).__init__()

        self.latent_dim = latent_dim
        self.delta = delta
        
        self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes
        neural_net_input = self.latent_dim*2 + genes + delta

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
            # nn.Linear(64, 1),
            # nn.Sigmoid()
        )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])


    def forward(self, image, sequence, genes):
        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Calculating delta
        delta_latent = torch.max(sequence, dim=1)[0] - torch.min(sequence, dim=1)[0]
        delta_latent = delta_latent.expand((-1, self.delta))

        # Fusion
        fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1), delta_latent), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = [torch.sigmoid(head(output)) for head in self.heads]

        return outputs

class EnsembleModel(nn.Module):
    def __init__(self, fusion):
        super(EnsembleModel, self).__init__()
        self.fusion = fusion
        self.fc = nn.Linear(4, 1)

    def forward(self, image, sequence, genes, igi_call):
        x = self.fusion(image, sequence, genes)
        x = torch.cat(x + [igi_call.view(-1, 1)], dim=1)
        x = torch.sigmoid(self.fc(x))
        return x
    
def run_logistic_regression(df, features, target, eval_df=None, eval_feats=None, output_save_name=None):
    relevant = features + [target]
    reldf = df[relevant]
    df2 = reldf.dropna()

    X = np.array(df2[features])
    y = np.array(df2[target])

    # Step 2: Run logistic regression
    model = sm.Logit(y, sm.add_constant(X)).fit()

    # Step 3: Report coefficients and significance
    print(model.summary())

    # For sklearn LogisticRegression model
    # model = LogisticRegression(max_iter=10000).fit(X, y)
    # coef_dict = {}
    # for coef, feat in zip(model.coef_[0], features):
    #     coef_dict[feat] = coef
    # print(coef_dict)

    # Step 4: Report AUC
    if (type(eval_df) != pd.DataFrame) or (eval_feats == None):
        y_prob = model.predict(sm.add_constant(X))
        auc = roc_auc_score(y, y_prob)
        print(f"AUC: {auc:.4f}\n")
    else:
        relevant = eval_feats + [target]
        reldf = eval_df[relevant]
        df2 = reldf.dropna()

        X = np.array(df2[eval_feats])
        y = np.array(df2[target])

        y_prob = model.predict(sm.add_constant(X))

        # if save_eval_df:
        # name = f"{target}_pred"
        eval_df['outputs'] = y_prob
        # time_now = time.time()
        eval_df[['curve_idx','outputs']].to_csv(output_save_name, index=False)

        # auc = roc_auc_score(y, y_prob)
        # print(f"AUC on test set: {auc:.4f}\n")

In [17]:
device = "cuda:2" if torch.cuda.is_available() else "cpu"
print(torch.version.cuda)
print(torch.__version__)
print(device)

11.8
2.1.0+cu118
cuda:2


In [18]:
with open('./data/groundtruth_df_curve_dict.pkl', 'rb') as file:
    curve_dict = pkl.load(file)
    
target_df = pd.read_csv('./data/groundtruth_df_target_data_split_v2.csv')  # Load your DataFrame here
target_df.loc[:,['groundtruth_target']] = 1*(target_df.groundtruth == 1)
#Define two error targets
target_df.loc[:,'Igi_call_quant'] = 1*(target_df.igi_call == 'Positive')
target_df['igi_fp'] = (target_df['Igi_call_quant'] > target_df['groundtruth_target']).astype(int)
target_df['igi_fn'] = (target_df['Igi_call_quant'] < target_df['groundtruth_target']).astype(int)


In [7]:
# known dilution dataset
known_df = pd.read_csv('./data/known_dilution_data.csv')

known_curve_dict = {}
for i in known_df.curve_idx.unique():
    df = known_df[known_df.curve_idx == i]
    known_curve_dict[i] = df['Fn'].to_list()

known_df['ct'] = (known_df
         .loc[known_df.drn >= known_df.threshold]
         .groupby(['curve_idx','well_position','target'])
         .cycle_no
         .transform(min))
known_df['igi_call'] = 'Positive'
known_df.loc[(known_df.ct.isna()) | ((known_df.target == 'RnaseP') & (known_df.ct >= 35)) |
          ((known_df.target != 'RnaseP') & (known_df.ct >= 40)),'igi_call'] = 'Negative'

known_df.loc[:,['groundtruth_target']] = 1
#Define two error targets
known_df.loc[:,'Igi_call_quant'] = 1*(known_df.igi_call == 'Positive')
known_df['igi_fp'] = (known_df['Igi_call_quant'] > known_df['groundtruth_target']).astype(int)
known_df['igi_fn'] = (known_df['Igi_call_quant'] < known_df['groundtruth_target']).astype(int)
known_df['split'] = 'test'

known_df = known_df[known_df.cycle_no == 40]

/tmp/ipykernel_3345550/141114837.py:13: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  .transform(min))


In [8]:
retest_df = pd.read_csv('./data/retest_df.csv')
retest_df['groundtruth_target'] = 1*(retest_df.final_patient_result == 'Positive')
retest_df['Igi_call_quant'] = 1*(retest_df.igi_call == 'Positive')
retest_df['igi_fp'] = (retest_df['Igi_call_quant'] > retest_df['groundtruth_target']).astype(int)
retest_df['igi_fn'] = (retest_df['Igi_call_quant'] < retest_df['groundtruth_target']).astype(int)
retest_df['split'] = 'test'

retest_curve_dict = {}
for i in retest_df.curve_idx.unique():
    df = retest_df[retest_df.curve_idx == i]
    retest_curve_dict[i] = df['Fn'].to_list()

retest_df = retest_df[retest_df.cycle_no == 40]

/tmp/ipykernel_3345550/772774594.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  retest_df = pd.read_csv('./data/retest_df.csv')


In [9]:
curve_df = pd.read_hdf('./data/data.h5', key='curve_data')
sample_info = pd.read_hdf('./data/data.h5', key='sample_info')
igi_gene_call = pd.read_hdf('./data/data.h5', key='igi_gene_call')

join_df = (curve_df
           .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
           .merge(igi_gene_call, how ='inner', on=['pcr_plate','sample_id','target']))

clinical_sample = join_df[(join_df.sample_type == 'Clinical Sample') & (join_df.retest_sample_id_1 == ' ')].copy()
clinical_sample_dict = {}
for i in clinical_sample.curve_idx.unique():
    df = clinical_sample[clinical_sample.curve_idx == i]
    clinical_sample_dict[i] = df['Fn'].to_list()

clinical_sample = clinical_sample[clinical_sample.cycle_no == 40]

dup_samples = clinical_sample.groupby('curve_idx').sample_id.nunique().reset_index()
remove_sample_id = clinical_sample.loc[(clinical_sample.curve_idx.isin(dup_samples.loc[dup_samples.sample_id > 1].curve_idx)) & (clinical_sample.record_type == 'Submitted Sample')].sample_id

clinical_sample = clinical_sample[~clinical_sample.sample_id.isin(remove_sample_id)]


In [10]:
clinical_sample['groundtruth_target'] = 1*(clinical_sample.final_patient_result == 'Positive')
clinical_sample['Igi_call_quant'] = 1*(clinical_sample.igi_call == 'Positive')
clinical_sample['igi_fp'] = 1
clinical_sample['igi_fn'] = 1
clinical_sample['split'] = 'test'

# retest_df['groundtruth_target'] = 1*(retest_df.final_patient_result == 'Positive')
# retest_df['Igi_call_quant'] = 1*(retest_df.igi_call == 'Positive')
# retest_df['igi_fp'] = (retest_df['Igi_call_quant'] > retest_df['groundtruth_target']).astype(int)
# retest_df['igi_fn'] = (retest_df['Igi_call_quant'] < retest_df['groundtruth_target']).astype(int)
# retest_df['split'] = 'test'

In [11]:
# Ensemble Logistic data

# Load the predictions
pred_df_test = pd.read_csv('./data/model_outputs/fusion_vit_delta64_test_pred_df.csv')
pred_df_val = pd.read_csv('./data/model_outputs/fusion_vit_delta64_val_pred_df.csv') 
pred_df_retest = pd.read_csv('./data/model_outputs/fusion_vit_delta64_retest_pred_df.csv')

target_df_test = target_df[target_df.split == 'test']
target_df_test = target_df_test.merge(pred_df_test, on = 'curve_idx', how='left')

target_df_val = target_df[target_df.split == 'val']
target_df_val = target_df_val.merge(pred_df_val, on = 'curve_idx', how='left')

target_df_retest = retest_df.merge(pred_df_retest, on = 'curve_idx', how='left')

# Step 1: One-hot encode the column "outputs" into 10 bins
encoder = KBinsDiscretizer(n_bins=10, encode='onehot-dense', strategy='uniform')
outputs_binned = encoder.fit_transform(target_df_val[['outputs']])
bin_edges = encoder.bin_edges_[0]
bin_names_val = ['outputs_bin_' + str(round(bin_edges[i],2)) + '_' + str(round(bin_edges[i+1],2)) for i in range(len(bin_edges)-1)]
target_df_val[bin_names_val] = pd.DataFrame(outputs_binned, columns=bin_names_val)

outputs_binned = encoder.transform(target_df_test[['outputs']])
bin_edges = encoder.bin_edges_[0]
bin_names_test = ['outputs_bin_' + str(round(bin_edges[i],2)) + '_' + str(round(bin_edges[i+1],2)) for i in range(len(bin_edges)-1)]
target_df_test[bin_names_test] = pd.DataFrame(outputs_binned, columns=bin_names_test)

outputs_binned = encoder.transform(target_df_retest[['outputs']])
bin_edges = encoder.bin_edges_[0]
bin_names_retest = ['outputs_bin_' + str(round(bin_edges[i],2)) + '_' + str(round(bin_edges[i+1],2)) for i in range(len(bin_edges)-1)]
target_df_retest[bin_names_retest] = pd.DataFrame(outputs_binned, columns=bin_names_retest)

# Features for regression
features_test = bin_names_test[1:] + ['Igi_call_quant']
features_val = bin_names_val[1:] + ['Igi_call_quant']
features_retest = bin_names_retest[1:] + ['Igi_call_quant']


/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:248: FutureWarning: In version 1.5 onwards, subsample=200_000 will be used by default. Set subsample explicitly to silence this warning in the mean time. Set subsample=None to disable subsampling explicitly.
  warnings.warn(


In [11]:
# # ###########################################
# # ## Save curves as images
# # ###########################################

# for idx in  tqdm(range(len(clinical_sample_dict.keys()))):
#     curve_idx = list(clinical_sample_dict.keys())[idx]
#     sequence = clinical_sample_dict[curve_idx][:40]

#     if not os.path.exists('./data/curve_imgs_axis/'):
#         os.makedirs('./data/curve_imgs_axis/')
#     plt.plot(sequence, linewidth=6)
#     #plt.axis('off')  # This will turn off the axis labels and ticks
#     plt.axis('on') 
#     plt.show()
#     plt.savefig(f'./data/curve_imgs_axis/curve_{curve_idx}.png')
#     plt.clf()

In [24]:
###########################################
## Get the right normalization values
###########################################

target_df_filtered = target_df[target_df['split']=='train']
curve_dict_filtered = {k: curve_dict[k] for k in curve_dict.keys() if k in target_df_filtered['curve_idx'].values}

mean_list = []
std_list = []

for key, curve in tqdm(curve_dict_filtered.items()):
    mean_curve = np.array(curve).mean().item()
    std_curve = np.array(curve).std().item()

    mean_list.append(mean_curve)
    std_list.append(std_curve)

norm_mean = np.array(mean_list).mean().item()
norm_std = np.array(std_list).mean().item()

###########################################
## Set-up data objects
###########################################

# TODO extract mean and std of sequences in train dataset

# Create Dataset and DataLoader
# train_dataset = ImageSequenceDataset(curve_dict, target_df,img_directory = '../data/curve_imgs_axis/', train=True, sequence_len=40,
#                                         mean=norm_mean, std = norm_std)
# val_dataset = ImageSequenceDataset(curve_dict, target_df,img_directory = './data/curve_imgs_axis/', split='val', sequence_len=40,
#                                     mean=norm_mean, std = norm_std)
# val_loader = DataLoader(val_dataset, batch_size=32, pin_memory=True, shuffle=True)

# data for fusion model
test_dataset = ImageSequenceDataset(curve_dict, target_df,img_directory = './data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
test_loader = DataLoader(test_dataset, batch_size=32, pin_memory=True, shuffle=True)

# data for fusion w gene
testwgene_dataset = ImageSequenceGeneDataset(curve_dict, target_df,img_directory = 'data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
testwgene_loader = DataLoader(testwgene_dataset, batch_size=32, pin_memory=True, shuffle=True)

# data for vit fusion w gene
trainVit_dataset = ViTFusionDataset(curve_dict, target_df,img_directory = 'data/curve_imgs_axis/', split='train', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
trainVit_loader = DataLoader(trainVit_dataset, batch_size=32, pin_memory=True, shuffle=True)

valvit_dataset = ViTFusionDataset(curve_dict, target_df,img_directory = 'data/curve_imgs_axis/', split='val', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
valvit_loader = DataLoader(valvit_dataset, batch_size=32, pin_memory=True, shuffle=True)

testvit_dataset = ViTFusionDataset(curve_dict, target_df,img_directory = 'data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
testvit_loader = DataLoader(testvit_dataset, batch_size=32, pin_memory=True, shuffle=True)

retestVit_dataset = ViTFusionDataset(retest_curve_dict, retest_df,img_directory = 'data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
retestVit_loader = DataLoader(retestVit_dataset, batch_size=32, pin_memory=True, shuffle=True)

knownVit_dataset = ViTFusionDataset(known_curve_dict, known_df,img_directory = 'data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
knownVit_loader = DataLoader(knownVit_dataset, batch_size=32, pin_memory=True, shuffle=True)

clinicalVit_dataset = ViTFusionDataset(clinical_sample_dict, clinical_sample,img_directory = 'data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
clinicalVit_loader = DataLoader(clinicalVit_dataset, batch_size=1, pin_memory=True, shuffle=True)

# data for covid ensemble
test_ensemble_dataset = EnsembleDataset(curve_dict, target_df,img_directory = 'data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
test_ensemble_loader = DataLoader(test_ensemble_dataset, batch_size=32, pin_memory=True, shuffle=True)

retest_ensemble_dataset = EnsembleDataset(retest_curve_dict, retest_df,img_directory = 'data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
retest_ensemble_loader = DataLoader(retest_ensemble_dataset, batch_size=32, pin_memory=True, shuffle=True)


100%|██████████| 13949/13949 [00:00<00:00, 50254.33it/s]


## Ensemble Logistic 

In [31]:
run_logistic_regression(target_df_val,features_val, 'groundtruth_target', 
                        target_df_test, features_test, 
                        './data/model_outputs/ensemble_logis_test_pred.csv')

run_logistic_regression(target_df_val,features_val, 'groundtruth_target', 
                        target_df_retest, features_retest, 
                        './data/model_outputs/ensemble_logis_retest_pred.csv')

Optimization terminated successfully.
         Current function value: 0.029854
         Iterations 11
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                 4632
Model:                          Logit   Df Residuals:                     4621
Method:                           MLE   Df Model:                           10
Date:                Sun, 05 Nov 2023   Pseudo R-squ.:                  0.9068
Time:                        18:30:04   Log-Likelihood:                -138.29
converged:                       True   LL-Null:                       -1483.2
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.7509      0.596     -7.969      0.000      -5.919      -3.582
x1             1.5723      0

## Ensemble Model

In [25]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3
genes = len(target_df['target'].unique())
delta_size = 64

fusion = ViTFusionModel(input_size, hidden_size, latent_dim, sequence_length, num_layers=num_layers, genes=genes, delta=delta_size)
# fusion.load_state_dict(torch.load('output/10_27_fusion_model_vit_delta64/best_model_no_pretrain.pth'))
model = EnsembleModel(fusion)
model.load_state_dict(torch.load('./output/fusion_model/10_31_ensemble_model_vit_delta64new.pth'))
model.to(device)  # If you are using GPU

RuntimeError: Error(s) in loading state_dict for EnsembleModel:
	Missing key(s) in state_dict: "fusion.vit.class_token", "fusion.vit.conv_proj.weight", "fusion.vit.conv_proj.bias", "fusion.vit.encoder.pos_embedding", "fusion.vit.encoder.layers.encoder_layer_0.ln_1.weight", "fusion.vit.encoder.layers.encoder_layer_0.ln_1.bias", "fusion.vit.encoder.layers.encoder_layer_0.self_attention.in_proj_weight", "fusion.vit.encoder.layers.encoder_layer_0.self_attention.in_proj_bias", "fusion.vit.encoder.layers.encoder_layer_0.self_attention.out_proj.weight", "fusion.vit.encoder.layers.encoder_layer_0.self_attention.out_proj.bias", "fusion.vit.encoder.layers.encoder_layer_0.ln_2.weight", "fusion.vit.encoder.layers.encoder_layer_0.ln_2.bias", "fusion.vit.encoder.layers.encoder_layer_0.mlp.0.weight", "fusion.vit.encoder.layers.encoder_layer_0.mlp.0.bias", "fusion.vit.encoder.layers.encoder_layer_0.mlp.3.weight", "fusion.vit.encoder.layers.encoder_layer_0.mlp.3.bias", "fusion.vit.encoder.layers.encode

#### Covid prediction

In [11]:
probs, curve_ids = [], []
model.eval()

for batch in tqdm(retest_ensemble_loader):
    images, sequences, gene, igi_call, _, _, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    gene = gene.to(device)
    igi_call = igi_call.to(device)

    out = model(images, sequences, gene, igi_call)
    probs.append(out.detach().cpu().numpy())

    curve_ids.append(ids)

test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(probs).squeeze()})
test_pred_df.to_csv('./data/model_outputs/ensemble_vit_fusion_delta64_retest_pred_df.csv', index = False)

100%|██████████| 128/128 [00:25<00:00,  4.94it/s]


#### IGI false

In [ ]:
probs, curve_ids = [], []
model.eval()

for batch in tqdm(retestVit_loader):
    images, sequences, gene, igi_call, igi_fp, _, _, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    gene = gene.to(device)
    igi_call = igi_call.to(device)

    out = model(images, sequences, gene, igi_call)
    probs.append(out.detach().cpu().numpy())

    curve_ids.append(ids)

## Fusion Model

In [6]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3

model = FusionModel(input_size, hidden_size, latent_dim, sequence_length, num_layers=num_layers)
model.load_state_dict(torch.load('./output/fusion_model/best_model_v2.pth'))
# model = ConvLSTM(input_size=input_size, conv_out_channels=32, kernel_size=3, hidden_size=50, output_size=2, seq_len=seq_len)
model.to(device)  # If you are using GPU


/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_L_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_L_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FusionModel(
  (effnet): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(3

In [43]:
test_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(test_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out, _ = model(images, sequences)
    test_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(test_outputs).squeeze()})
test_pred_df.to_csv('./data/model_outputs/fusion_test_pred_df.csv', index = False)

100%|██████████| 193/193 [00:37<00:00,  5.15it/s]


In [7]:
vision_latents, seq_latents = [], []
curve_ids = []
model.eval()
for batch in tqdm(test_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out, vision_latent, seq_latent = model(images, sequences)
    vision_latents.append(vision_latent.cpu().detach().numpy())
    seq_latents.append(seq_latent.cpu().detach().numpy())
    curve_ids.append(ids)

# test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(test_outputs).squeeze()})
# test_pred_df.to_csv('./data/model_outputs/fusion_test_pred_df.csv', index = False)

100%|██████████| 193/193 [00:38<00:00,  5.06it/s]


In [8]:
# test_latent_np = np.concatenate(test_latent)
with open('./data/model_outputs/fusion_vision_latent.pkl', 'wb') as file:
    pkl.dump({'latent': np.concatenate(vision_latents),
              'curve_ids': np.concatenate(curve_ids)}, file)
    
with open('./data/model_outputs/fusion_seq_latent.pkl', 'wb') as file:
    pkl.dump({'latent': np.concatenate(seq_latents),
              'curve_ids': np.concatenate(curve_ids)}, file)


## ViT Fusion with gene

In [25]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3
genes = len(target_df['target'].unique())
delta_size = 64

model = ViTFusionModel(input_size, hidden_size, latent_dim, sequence_length, num_layers=num_layers, genes=genes, delta=delta_size)
model.load_state_dict(torch.load('./output/fusion_model/10_27_fusion_model_vit_delta64.pth')) #./output/fusion_model/fusion_vit_delta64.pth'))
#10_31_ensemble_model_vit_delta64new.pth'))
# model = ConvLSTM(input_size=input_size, conv_out_channels=32, kernel_size=3, hidden_size=50, output_size=2, seq_len=seq_len)
model.to(device)  # If you are using GPU


ViTFusionModel(
  (vit): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          (ln_1): La

In [26]:
probs, fusion_space, igi_fp, igi_fn = [], [], [], []
curve_ids = []
model.eval()
for batch in tqdm(knownVit_loader):
    images, sequences, gene, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    gene = gene.to(device)

    try:
        out = model(images, sequences, gene)
    except:
        print(images.shape, sequences.shape, gene.shape, ids)
    probs.append(out[0].detach().cpu().numpy())
    # fusion_space.append(fusion.detach().cpu().numpy())
    igi_fp.append(out[1].detach().cpu().numpy())
    igi_fn.append(out[2].detach().cpu().numpy())

    curve_ids.append(ids)



100%|██████████| 9/9 [00:01<00:00,  4.90it/s]


In [27]:
test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 
                             'outputs': np.concatenate(probs).squeeze(),
                             'igi_fp': np.concatenate(igi_fp).squeeze(),
                             'igi_fn': np.concatenate(igi_fn).squeeze()})
test_pred_df.to_csv('./data/model_outputs/fusion_vit_delta64_known_pred_df.csv', index = False)

In [14]:
with open('./data/model_outputs/ensemble_fusion_vit_delta64_retest_latent.pkl', 'wb') as file:
    pkl.dump({'latent': np.concatenate(fusion_space),
              'curve_ids': np.concatenate(curve_ids)}, file)
    

## Fusion with gene

In [92]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3
genes = len(target_df['target'].unique())

model = FusionwGeneModel(input_size, hidden_size, latent_dim, sequence_length, num_layers=num_layers, genes=genes)
model.load_state_dict(torch.load('./output/fusion_model/fusion_w_gene.pth'))
# model = ConvLSTM(input_size=input_size, conv_out_channels=32, kernel_size=3, hidden_size=50, output_size=2, seq_len=seq_len)
model.to(device)  # If you are using GPU


FusionwGeneModel(
  (effnet): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNor

In [93]:
probs, igi_fp, igi_fn = [], [], []
curve_ids = []
model.eval()
for batch in tqdm(testwgene_loader):
    images, sequences, gene, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    gene = gene.to(device)

    out = model(images, sequences, gene)
    probs.append(out[0].detach().cpu().numpy())
    igi_fp.append(out[1].detach().cpu().numpy())
    igi_fn.append(out[2].detach().cpu().numpy())

    curve_ids.append(ids)



100%|██████████| 193/193 [00:38<00:00,  5.04it/s]


In [94]:
test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 
                             'outputs': np.concatenate(probs).squeeze(),
                             'igi_fp': np.concatenate(igi_fp).squeeze(),
                             'igi_fn': np.concatenate(igi_fn).squeeze()})
test_pred_df.to_csv('./data/model_outputs/fusion_w_gene_test_pred_df.csv', index = False)

In [90]:
probs, igi_fp, igi_fn, latents = [], [], [], []
curve_ids = []
model.eval()
for batch in tqdm(testwgene_loader):
    images, sequences, gene, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    gene = gene.to(device)

    out, latent = model(images, sequences, gene)
    probs.append(out[0].detach().cpu().numpy())
    igi_fp.append(out[1].detach().cpu().numpy())
    igi_fn.append(out[2].detach().cpu().numpy())
    latents.append(latent.detach().cpu().numpy())
    curve_ids.append(ids)



100%|██████████| 193/193 [00:38<00:00,  5.01it/s]


In [89]:
with open('./data/model_outputs/fusion_w_gene_latent.pkl', 'wb') as file:
    pkl.dump({'latent': np.concatenate(latents),
              'curve_ids': np.concatenate(curve_ids)}, file)
    

## Fusion Model without pretrain

In [2]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3


model = FusionModel(input_size, hidden_size, latent_dim, sequence_length, num_layers=num_layers)
model.load_state_dict(torch.load('./output/fusion_model/fusion_w_no_pretain.pth'))
# model = ConvLSTM(input_size=input_size, conv_out_channels=32, kernel_size=3, hidden_size=50, output_size=2, seq_len=seq_len)
model.to(device)  # If you are using GPU

NameError: name 'target_df' is not defined

In [8]:
test_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(test_loader):
    images, sequences, gene, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    gene = gene.to(device)

    out = model(images, sequences, gene)
    test_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(test_outputs).squeeze()})
test_pred_df.to_csv('./data/model_outputs/fusion_wo_pretrain_test_pred_df.csv', index = False)

100%|██████████| 193/193 [00:38<00:00,  5.03it/s]


## Image only model

In [7]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3
num_epoch = 10

model = ImageModel(input_size, hidden_size, latent_dim)
model.load_state_dict(torch.load('./output/image_model/image_best_model_ep50.pth', map_location=torch.device(device)))
model.to(device) 

/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_L_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_L_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ImageModel(
  (effnet): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(32

In [8]:

val_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(val_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out = model(images, sequences)
    val_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

val_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(val_outputs).squeeze()})
val_pred_df.to_csv('./data/model_outputs/img_only_ep50_val_pred_df.csv', index = False)

100%|██████████| 145/145 [00:28<00:00,  5.02it/s]


In [9]:

test_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(test_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out = model(images, sequences)
    test_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(test_outputs).squeeze()})
test_pred_df.to_csv('./data/img_only_ep50_test_pred_df.csv', index = False)

100%|██████████| 193/193 [00:37<00:00,  5.14it/s]


## Sequence only model

In [10]:
   
class SequenceModel(nn.Module):
    def __init__(self, input_size, hidden_size, latent_dim, sequence_length):
        super(SequenceModel, self).__init__()

        self.latent_dim = latent_dim
        
        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, 512)

        # FC Layers
        self.fc = nn.Sequential(
            nn.Linear(self.latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, image, sequence):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        output = self.fc(seq_latent)
        return output
    

In [17]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3
num_epoch = 50

model = SequenceModel(input_size, hidden_size, latent_dim, sequence_length)
model.load_state_dict(torch.load('./output/seq_model/seq_best_model_ep50.pth', map_location=torch.device(device)))
model.to(device)  # If you are using GPU

SequenceModel(
  (lstm): LSTM(1, 512, num_layers=3, batch_first=True)
  (lstm_fc): Linear(in_features=512, out_features=512, bias=True)
  (fc): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=1, bias=True)
    (7): Sigmoid()
  )
)

In [18]:

test_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(test_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out = model(images, sequences)
    test_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(test_outputs).squeeze()})
test_pred_df.to_csv('./data/model_outputs/seq_only_test_pred_df.csv', index = False)

100%|██████████| 193/193 [00:31<00:00,  6.22it/s]


In [19]:

val_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(val_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out = model(images, sequences)
    val_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

val_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(val_outputs).squeeze()})
val_pred_df.to_csv('./data/model_outputs/seq_only_val_pred_df.csv', index = False)

100%|██████████| 145/145 [00:22<00:00,  6.33it/s]
